In [1]:
import pdfplumber
from difflib import SequenceMatcher
import re
import pandas as pd
from IPython.display import display

In [2]:
# text helpers

def normalize_text(text):
    if text is None:
        return ""
    return " ".join(str(text).lower().split())


def row_to_text(row):
    return " | ".join(normalize_text(cell) for cell in row if cell is not None)


def table_to_text(table):
    return "\n".join(row_to_text(row) for row in table if row)

In [3]:
# clean rows

def clean_table_rows(table):
    cleaned = []
    for row in table:
        if not row:
            continue

        cleaned_row = []
        has_nonempty = False

        for cell in row:
            cell_text = "" if cell is None else str(cell).strip()
            cleaned_row.append(cell_text)
            if cell_text != "":
                has_nonempty = True

        if has_nonempty:
            cleaned.append(cleaned_row)

    return cleaned

In [4]:
# title guess helpers

def extract_candidate_titles_from_page_text(page_text):
    """
    Try to find lines like:
    Table 1: ...
    Table 34: ...
    Figure 2: ...
    """
    if not page_text:
        return []

    lines = [line.strip() for line in page_text.split("\n") if line.strip()]
    title_candidates = []

    pattern = re.compile(r"^(table|figure)\s+\d+[:.]?", re.IGNORECASE)

    for line in lines:
        if pattern.match(line):
            title_candidates.append(line)

    return title_candidates


def guess_table_title(cleaned_table, page_text):
    """
    - if page has one table title candidate, use it
    - if multiple, try matching table text to candidate text
    """
    candidates = extract_candidate_titles_from_page_text(page_text)
    if not candidates:
        return None

    if len(candidates) == 1:
        return candidates[0]

    table_text = table_to_text(cleaned_table)
    table_text_norm = normalize_text(table_text)

    best_title = None
    best_score = -1

    for c in candidates:
        c_norm = normalize_text(c)
        score = SequenceMatcher(None, c_norm, table_text_norm[:1000]).ratio()
        if score > best_score:
            best_score = score
            best_title = c

    return best_title

In [5]:
# extraction

def extract_pdf_tables(file_path, table_settings=None, verbose=True):
    extracted_tables = []

    default_table_settings = {
        "vertical_strategy": "lines",
        "horizontal_strategy": "lines",
        "snap_tolerance": 3,
        "join_tolerance": 3,
        "intersection_tolerance": 3,
    }

    settings = table_settings if table_settings is not None else default_table_settings

    try:
        with pdfplumber.open(file_path) as pdf:
            if verbose:
                print(f"Successfully loaded PDF with {len(pdf.pages)} pages.\n")

            for page_num, page in enumerate(pdf.pages, start=1):
                page_text_raw = page.extract_text() or ""
                page_text = normalize_text(page_text_raw)

                try:
                    tables = page.extract_tables(table_settings=settings)
                except Exception as e:
                    if verbose:
                        print(f"[Warning] Failed extracting tables on page {page_num}: {e}")
                    continue

                if not tables:
                    continue

                if verbose:
                    print(f"=== Found {len(tables)} table(s) on Page {page_num} ===")

                for table_num, table in enumerate(tables, start=1):
                    cleaned_table = clean_table_rows(table)

                    if not cleaned_table:
                        continue

                    n_rows = len(cleaned_table)
                    n_cols = max((len(r) for r in cleaned_table), default=0)
                    table_text_raw = table_to_text(cleaned_table)
                    table_text = normalize_text(table_text_raw)

                    title_guess = guess_table_title(cleaned_table, page_text_raw)

                    extracted_tables.append({
                        "page": page_num,
                        "table_num": table_num,
                        "rows": cleaned_table,
                        "table_text": table_text,
                        "page_text": page_text,
                        "title_guess": title_guess,
                        "n_rows": n_rows,
                        "n_cols": n_cols,
                    })

        return extracted_tables

    except FileNotFoundError:
        print(f"Error: Could not find the file at {file_path}")
        return []
    except Exception as e:
        print(f"An error occurred while opening the PDF: {e}")
        return []

In [6]:
# print full table

def print_table_full(table_info):
    print("\n" + "=" * 90)
    print(
        f"Page {table_info['page']} | "
        f"Table {table_info['table_num']} | "
        f"Rows: {table_info['n_rows']} | Cols: {table_info['n_cols']}"
    )
    print(f"title_guess: {table_info.get('title_guess')}")
    print("=" * 90)

    for row in table_info["rows"]:
        print(row)

In [7]:
pdf_path = "2025-08_Gaza Joint WASH Assessment Report.pdf"
all_tables = extract_pdf_tables(pdf_path, verbose=True)
print(f"Total extracted tables: {len(all_tables)}")

Successfully loaded PDF with 47 pages.

=== Found 1 table(s) on Page 14 ===
=== Found 2 table(s) on Page 16 ===
=== Found 1 table(s) on Page 17 ===
=== Found 1 table(s) on Page 19 ===
=== Found 3 table(s) on Page 20 ===
=== Found 3 table(s) on Page 21 ===
=== Found 1 table(s) on Page 22 ===
=== Found 2 table(s) on Page 23 ===
=== Found 2 table(s) on Page 24 ===
=== Found 3 table(s) on Page 25 ===
=== Found 1 table(s) on Page 27 ===
=== Found 3 table(s) on Page 28 ===
=== Found 2 table(s) on Page 29 ===
=== Found 1 table(s) on Page 30 ===
=== Found 2 table(s) on Page 31 ===
=== Found 2 table(s) on Page 32 ===
=== Found 2 table(s) on Page 33 ===
=== Found 2 table(s) on Page 34 ===
=== Found 1 table(s) on Page 35 ===
=== Found 2 table(s) on Page 36 ===
=== Found 4 table(s) on Page 37 ===
=== Found 3 table(s) on Page 38 ===
Total extracted tables: 44


In [ ]:
# 1) INDICATOR MAP
INDICATOR_MAP = {
    "% of households with less than 6 lppd of drinking water": [
        "less than 6 lppd of drinking water",
        "6 lppd of drinking water"
    ],
    "% of households with less than 9 lppd of domestic water": [
        "less than 9 lppd of domestic water",
        "9 lppd of domestic water"
    ],
    "% of households with less than 15 lppd of water for all purposes": [
        "less than 15 lppd of water for all purposes",
        "15 lppd of water for all purposes"
    ],
    "% of households practicing open defecation": [
        "households practicing open defecation",
        "practicing open defecation"
    ],
    "% of households with unimproved sanitation facilities": [
        "households with unimproved sanitation facilities",
        "unimproved sanitation facilities"
    ],
    "% of households with basic sanitation facilities": [
        "households with basic sanitation facilities",
        "basic sanitation facilities"
    ],
    "% of household with safely managed sanitation facilities": [
        "household with safely managed sanitation facilities",
        "households with safely managed sanitation facilities",
        "safely managed sanitation facilities"
    ],
    "% of households with sewage or feces within 10 m": [
        "households with sewage or feces within 10 m",
        "sewage or feces within 10 m",
        "sewage or feces within 10m"
    ],
    "% of household with waste piled within 10 m": [
        "household with waste pilled wihin 10 m",
        "households with waste pilled wihin 10 m",
        "waste pilled wihin 10 m",
        "waste piled within 10 m",
        "waste piled within 10m"
    ],
    "% of households NOT having soap at accommodation": [
        "households not having soap at accommodation",
        "not having soap at accommodation",
        "not having soap"
    ],
    "% of assessed households reported that at least one household member had lice in the 30 days prior to data collection.": [
        "households reported that at least one household member had lice in the 30 days prior to data collection",
        "had lice in the 30 days prior to data collection",
        "lice in the 30 days prior to data collection"
    ],
    "% of assessed households reported that at least one household member had bedbugs in the 30 days prior to data collection.": [
        "households reported that at least one household member had bedbugs in the 30 days prior to data collection",
        "had bedbugs in the 30 days prior to data collection",
        "bedbugs in the 30 days prior to data collection"
    ],
    "% of assessed households reported that at least one household member had skin rashes or scabies in the 30 days prior to data collection.": [
        "households reported that at least one household member had skin rashes or scabies in the 30 days prior to data collection",
        "skin rashes or scabies in the 30 days prior to data collection",
        "skin rashes or scabies"
    ],
}


# 2) HELPERS
def clean_text(x):
    if x is None:
        return ""
    x = str(x).replace("\n", " ").replace("\r", " ").strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

def clean_value(x):
    if x is None:
        return pd.NA
    x = str(x).strip()
    return x if x != "" else pd.NA

def rows_equal(values1, values2):
    return [clean_text(v) for v in values1] == [clean_text(v) for v in values2]

def format_table_label(table_num):
    return f"Table {table_num}" if pd.notna(table_num) else pd.NA

def exact_value_like(x):
    if x is None:
        return False
    s = str(x).strip().lower()
    if s == "":
        return False
    return bool(re.fullmatch(r"-|na|n/a|\d+(\.\d+)?%?|\.\d+%?", s))

def values_look_like_data(values):
    return sum(exact_value_like(v) for v in values) >= 2

def cell_too_long(x, max_len=25):
    if x is None:
        return False
    return len(str(x).strip()) > max_len

def header_cell_like(x):
    s = clean_text(x)
    if s == "":
        return False
    # looks like date / period / time header
    patterns = [
        r"\baugust\b|\bjanuary\b|\bfebruary\b|\bmarch\b|\bapril\b|\bmay\b|\bjune\b|\bjuly\b|\bseptember\b|\boctober\b|\bnovember\b|\bdecember\b",
        r"\b20\d{2}\b",
        r"\bq[1-4]\b",
        r"\bweek\b|\bmonth\b|\byear\b|\bdate\b|\bperiod\b"
    ]
    return any(re.search(p, s) for p in patterns)

def infer_headers(rows, row_idx, indicator_pos, n_values=3):
    """
    Try to detect header names from rows above the matched row.
    Prefer the nearest previous row whose cells after indicator_pos look like dates/periods.
    """
    for lookback in range(1, 4):
        prev_idx = row_idx - lookback
        if prev_idx < 0:
            break

        prev_row = ["" if x is None else str(x).strip() for x in rows[prev_idx]]
        candidate_headers = prev_row[indicator_pos + 1: indicator_pos + 1 + n_values]
        candidate_headers = candidate_headers + [""] * (n_values - len(candidate_headers))

        if sum(header_cell_like(h) for h in candidate_headers) >= 2:
            return [h if str(h).strip() != "" else f"value_{i+1}" for i, h in enumerate(candidate_headers)]

    return [f"value_{i+1}" for i in range(n_values)]


# 3) SEARCH ALL TABLES FOR ALL INDICATORS
all_matches = []

for table_info in all_tables:
    page = table_info.get("page")
    table_num = table_info.get("table_num")
    title_guess = table_info.get("title_guess")

    rows = table_info.get("rows", [])
    rows = [r for r in rows if any(clean_text(c) != "" for c in r)]

    for row_idx, row in enumerate(rows):
        row_clean = ["" if x is None else str(x).strip() for x in row]
        row_text = clean_text(" | ".join(row_clean))

        for desired_name, phrases in INDICATOR_MAP.items():
            matched = any(clean_text(phrase) in row_text for phrase in phrases)
            if not matched:
                continue

            indicator_pos = None
            for i, cell in enumerate(row_clean):
                cell_text = clean_text(cell)
                if any(clean_text(phrase) in cell_text for phrase in phrases):
                    indicator_pos = i
                    break

            if indicator_pos is None:
                continue

            candidate_values = row_clean[indicator_pos + 1: indicator_pos + 4]
            candidate_values = candidate_values + [""] * (3 - len(candidate_values))

            if any(cell_too_long(v) for v in candidate_values):
                continue
            if not values_look_like_data(candidate_values):
                continue

            detected_headers = infer_headers(rows, row_idx, indicator_pos, n_values=3)

            all_matches.append({
                "indicator": desired_name,
                "matched_row_text": " | ".join(row_clean),
                "page": page,
                "table_num": table_num,
                "table_label": format_table_label(table_num),
                "title_guess": title_guess,
                "row_idx": row_idx,
                "values": candidate_values,
                "headers": detected_headers,
                "full_row": row_clean
            })


# 4) DETERMINE OUTPUT COLUMNS
default_headers = ["value_1", "value_2", "value_3"]
detected_output_headers = None

for m in all_matches:
    if m["headers"] and len(m["headers"]) == 3:
        detected_output_headers = m["headers"]
        break

if detected_output_headers is None:
    detected_output_headers = default_headers

# make them unique and non-empty
seen = set()
final_headers = []
for i, h in enumerate(detected_output_headers):
    h = str(h).strip() if h is not None else ""
    if h == "":
        h = f"value_{i+1}"
    original = h
    j = 2
    while h in seen:
        h = f"{original}_{j}"
        j += 1
    seen.add(h)
    final_headers.append(h)


# 5) BUILD FINAL OUTPUT TABLE
first_unique_records = []
inconsistencies = []

for desired_name in INDICATOR_MAP.keys():
    indicator_matches = [m for m in all_matches if m["indicator"] == desired_name]

    if len(indicator_matches) == 0:
        record = {
            "indicator": desired_name,
            final_headers[0]: pd.NA,
            final_headers[1]: pd.NA,
            final_headers[2]: pd.NA,
            "source_page": pd.NA,
            "source_table": pd.NA
        }
        first_unique_records.append(record)
        continue

    first_match = indicator_matches[0]

    record = {
        "indicator": desired_name,
        final_headers[0]: clean_value(first_match["values"][0]),
        final_headers[1]: clean_value(first_match["values"][1]),
        final_headers[2]: clean_value(first_match["values"][2]),
        "source_page": first_match["page"],
        "source_table": first_match["table_label"]
    }
    first_unique_records.append(record)

    for other_match in indicator_matches[1:]:
        if not rows_equal(first_match["values"], other_match["values"]):
            inconsistencies.append({
                "indicator": desired_name,
                "reference_page": first_match["page"],
                "reference_table": first_match["table_label"],
                "reference_values": dict(zip(final_headers, first_match["values"])),
                "inconsistent_page": other_match["page"],
                "inconsistent_table": other_match["table_label"],
                "inconsistent_values": dict(zip(final_headers, other_match["values"]))
            })


# 6) FINAL TABLE
final_df = pd.DataFrame(first_unique_records)

final_long_df = final_df.melt(
    id_vars=["indicator", "source_page", "source_table"],
    var_name="date",
    value_name="value"
)

# keep missing values as true missing inside Python
final_long_df["value"] = final_long_df["value"].replace(["", "-", "—"], pd.NA)

# split "August 2024" into separate month and year columns
date_parts = final_long_df["date"].astype(str).str.extract(r"^\s*([A-Za-z]+)\s+(\d{4})\s*$")
final_long_df["month"] = date_parts[0]
final_long_df["year"] = pd.to_numeric(date_parts[1], errors="coerce").astype("Int64")

# add source column
source_name = "Gaza Joint WASH Assessment Round 3.pdf"
final_long_df["source_dataset"] = source_name

# reorder columns with source at the end
final_long_df = final_long_df[[
    "indicator", "month", "year", "value", "source_page", "source_table", "source_dataset"
]]

print("\nFINAL TABLE")
display(final_long_df)


# 7) CONSISTENCY CONCLUSION
if len(inconsistencies) == 0:
    print("\nConsistency conclusion:")
    print("No inconsistent data found in the file.")
else:
    inconsistency_df = pd.DataFrame(inconsistencies)

    print("\nConsistency conclusion:")
    print("Found inconsistent data in the file for the following indicator(s):")

    for _, row in inconsistency_df.iterrows():
        print(
            f"- Indicator: {row['indicator']}\n"
            f"  Reference: page {row['reference_page']}, {row['reference_table']}, values={row['reference_values']}\n"
            f"  Inconsistent: page {row['inconsistent_page']}, {row['inconsistent_table']}, values={row['inconsistent_values']}"
        )

    print("\nDetailed inconsistency table:")
    display(inconsistency_df)


# 8) SAVE OUTPUTS
final_output_csv = "gaza_joint_wash_assessment_data.csv"
consistency_output_csv = "indicator_consistency_check.csv"

try:
    export_df = final_long_df.copy()
    export_df["value"] = export_df["value"].fillna("Not reported")

    export_df.to_csv(final_output_csv, encoding="utf-8-sig", index=False)
    pd.DataFrame(inconsistencies).to_csv(consistency_output_csv, encoding="utf-8-sig", index=False)

    print(f"\nSaved final table to: {final_output_csv}")
    print(f"Saved consistency report to: {consistency_output_csv}")

except PermissionError:
    print("\nCould not save because one of the output files is open. Close the file and rerun.")


FINAL TABLE


,indicator,month,year,value,source_page,source_table,source_dataset
0,% of households with less than 6 lppd of drink...,August,2024,64%,17,Table 1,Gaza Joint WASH Assessment Round 3.pdf
1,% of households with less than 9 lppd of domes...,August,2024,46%,17,Table 1,Gaza Joint WASH Assessment Round 3.pdf
2,% of households with less than 15 lppd of wate...,August,2024,46%,17,Table 1,Gaza Joint WASH Assessment Round 3.pdf
3,% of households practicing open defecation,August,2024,6%,17,Table 1,Gaza Joint WASH Assessment Round 3.pdf
4,% of households with unimproved sanitation fac...,August,2024,<NA>,17,Table 1,Gaza Joint WASH Assessment Round 3.pdf
5,% of households with basic sanitation facilities,August,2024,<NA>,17,Table 1,Gaza Joint WASH Assessment Round 3.pdf
6,% of household with safely managed sanitation ...,August,2024,0%,17,Table 1,Gaza Joint WASH Assessment Round 3.pdf
7,% of households with sewage or feces within 10 m,August,2024,79%,17,Table 1,Gaza Joint WASH Assessment Round 3.pdf
8,% of household with waste piled within 10 m,August,2024,54%,17,Table 1,Gaza Joint WASH Assessment Round 3.pdf
9,% of households NOT having soap at accommodation,August,2024,76%,17,Table 1,Gaza Joint WASH Assessment Round 3.pdf



Consistency conclusion:
No inconsistent data found in the file.

Saved final table to: indicator_first_unique_values.csv
Saved consistency report to: indicator_consistency_check.csv
